In [1]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Dense, LSTM, Dropout
from tensorflow.keras.callbacks import EarlyStopping
import keras_tuner
from tensorflow.keras.optimizers import Adam

In [5]:
# Read datasets
data = pd.read_csv("../data/04_processed/train.csv")

# Split to training and validation set
train = data[data["year"] < 2013].copy()
val = data[data["year"] >= 2013].copy()

In [53]:
# Normalize x,y coords
train["x_norm"] = train["x_coord"] / train["x_coord"].max()
train["y_norm"] = train["y_coord"] / train["x_coord"].max()

val["x_norm"] = val["x_coord"] / train["y_coord"].max() 
val["y_norm"] = val["y_coord"] / train["y_coord"].max() 

In [54]:
train = train.sort_values(by=["x_coord", "y_coord", "date_time"])
val = val.sort_values(by=["x_coord", "y_coord", "date_time"])

In [55]:
# Create lag feature
train["lag_27"] = train.groupby(["x_coord","y_coord"])["water_percentage"].shift(27)
val["lag_27"] = val.groupby(["x_coord","y_coord"])["water_percentage"].shift(27)

train = train.dropna(subset=["lag_27"])
val = val.dropna(subset=["lag_27"])

In [56]:
def create_water_data(data, lookback):
    dataX, dataY = [], []
    # group data per grid and per year
    group_data = data.groupby(["x_coord", "y_coord", "year"])

    for _, grid_data_per_year in group_data:
        grid_data_per_year = grid_data_per_year.sort_values("date_time")
        data_values = grid_data_per_year[["water_percentage","lag_27","x_norm","y_norm"]].values
        for i in range(len(data_values) - lookback):
            a = data_values[i : (i + lookback), :]
            dataX.append(a)
            dataY.append(data_values[i + lookback,0])
        
    return np.array(dataX),np.array(dataY)

In [57]:
# Drop irrelevant columns for training/validation
train = train[["year","water_percentage","lag_27","x_coord","y_coord","date_time","x_norm","y_norm"]]
val = val[["year","water_percentage","lag_27","x_coord","y_coord","date_time","x_norm","y_norm"]]

In [58]:
# Create data with lookback=4
lookback = 4
X_train, y_train = create_water_data(train,lookback)
X_val, y_val = create_water_data(val,lookback)

In [59]:
X_train.shape

(27324, 4, 4)

In [60]:
def build_model(hp):
    model = Sequential()
    model.add(Input(shape=(4, 4)))
    # 1st LSTM Layer: Changed input_shape to (lookback, 2)
    model.add(LSTM(
        units=hp.Int("units_1", min_value=32, max_value=256, step=32),
        activation="tanh", return_sequences=True))
    model.add(Dropout(hp.Float("dropout_1", 0.1, 0.4, step=0.1)))
    # 2nd LSTM Layer: Processes the sequence
    model.add(LSTM(
        units=hp.Int("units_2", min_value=16, max_value=128, step=16),
        activation="tanh", return_sequences=False))
    model.add(Dropout(hp.Float("dropout_2", 0.1, 0.4, step=0.1)))
    model.add(Dense(1))
    
    # Tuning the Learning Rate
    lr = hp.Choice("learning_rate", values=[1e-2, 1e-3, 1e-4])
    model.compile(optimizer=Adam(learning_rate=lr), loss="mean_squared_error")
    return model

In [62]:
# Initialize Bayesian Optimizer
tuner = keras_tuner.BayesianOptimization(
    build_model,
    objective="val_loss",
    max_trials=30,
    executions_per_trial=1,
    seed=123,
    directory="tuning_results",
    project_name="water_level_prediction_lstm")

In [63]:
print("Starting hyperparameter search...")
early_stop = EarlyStopping(monitor="val_loss", patience=5)
tuner.search(
    X_train, y_train,
    epochs=50,
    validation_data=(X_val, y_val), # This enforces the temporal split
    verbose=1,
    callbacks=[early_stop]
)

Trial 30 Complete [00h 01m 44s]
val_loss: 0.006348788738250732

Best val_loss So Far: 0.005982415284961462
Total elapsed time: 00h 26m 24s


In [44]:
best_hps = tuner.get_best_hyperparameters(num_trials=1)[0]
best_model = tuner.hypermodel.build(best_hps)

print(f"""
Optimal Configuration found:
- Units 1: {best_hps.get("units_1")}
- Units 2: {best_hps.get("units_2")}
- Learning Rate: {best_hps.get('learning_rate')}
- Dropout 1: {best_hps.get("dropout_1")}
- Dropout 2: {best_hps.get("dropout_2")}
""")


Optimal Configuration found:
- Units 1: 160
- Units 2: 16
- Learning Rate: 0.01
- Dropout 1: 0.1
- Dropout 2: 0.1



## Hyper parameter tuning - GRU

In [64]:
from tensorflow.keras.layers import GRU

def build_gru_model(hp):
    model = Sequential()
    model.add(Input(shape=(4, 4)))
    # Layer 1: Must return sequences
    model.add(GRU(units=hp.Int("units_1", 32, 256), 
                  activation='tanh', 
                  return_sequences=True))
    model.add(Dropout(hp.Float("dropout_1", 0.1, 0.4, step=32)))
    
    # Layer 2: Processes the sequence from Layer 1
    model.add(GRU(units=hp.Int("units_2", 16, 128), 
                  activation='tanh', 
                  return_sequences=False))
    model.add(Dropout(hp.Float("dropout_2", 0.1, 0.4, step=16)))
    
    model.add(Dense(1))
    
    model.compile(
        optimizer=Adam(hp.Choice("learning_rate", [1e-2, 1e-3, 1e-4])),
        loss='mse'
    )
    return model

In [65]:
gru_tuner = keras_tuner.BayesianOptimization(
    build_gru_model,
    objective="val_loss",
    max_trials=30,
    executions_per_trial=1,
    seed=123,
    directory="tuning_results",
    project_name="water_level_prediction_gru")

In [66]:
print("Starting hyperparameter search...")
early_stop = EarlyStopping(monitor='val_loss', patience=5)
gru_tuner.search(
    X_train, y_train,
    epochs=50,
    validation_data=(X_val, y_val), # This enforces the temporal split
    verbose=1,
    callbacks=[early_stop]
)

Trial 30 Complete [00h 00m 57s]
val_loss: 0.0061662159860134125

Best val_loss So Far: 0.005987432319670916
Total elapsed time: 00h 17m 44s


In [67]:
best_gru_hps = gru_tuner.get_best_hyperparameters(num_trials=1)[0]
best_gru_model = gru_tuner.hypermodel.build(best_gru_hps)

print(f"""
Optimal Configuration found:
- Units 1: {best_gru_hps.get('units_1')}
- Units 2: {best_gru_hps.get('units_2')}
- Learning Rate: {best_gru_hps.get('learning_rate')}
- Dropout 1: {best_gru_hps.get("dropout_1")}
- Dropout 2: {best_gru_hps.get("dropout_2")}
""")


Optimal Configuration found:
- Units 1: 250
- Units 2: 36
- Learning Rate: 0.001
- Dropout 1: 0.1
- Dropout 2: 0.1

